# 02 · Explore the patterns: what relates to sepsis and to death?
**BT3041 Sepsis project** · a first look across Steps 3, 4, 6 and 7 of the manual

**Run `01_load_and_qc.ipynb` first**: this notebook reads its cleaned files from `data/processed/`.

| Section | Question | Course tools |
|---|---|---|
| 1 | Who is in the study? | descriptive statistics |
| 2 | Do age, sex, diabetes, hospital, endotype, infection source or batch relate to 28-day death? | Welch t, Mann–Whitney, chi-square, Fisher, logistic regression |
| 3 | Which genes differ between patients who died and survived, and between sepsis and healthy? | Welch t per gene, Benjamini–Hochberg FDR |
| 4 | Is there structure in the data? Is batch a problem? | PCA, R² |
| 5 | Figures | |

**How to run:** Runtime → Run all (~1–2 minutes). *Expected* lines show the numbers we got.

In [ ]:
# ---- SETUP CELL: run this first ----
import os
try:
    from google.colab import drive              # running in Google Colab
    drive.mount('/content/drive')               # click the link, allow access
    BASE = "/content/drive/MyDrive/BT3041_Sepsis"
except ImportError:                              # running in Jupyter on a laptop
    BASE = os.environ.get("BT3041_BASE", os.path.abspath(".."))

RAW  = os.path.join(BASE, "data", "raw")
PROC = os.path.join(BASE, "data", "processed")
FIG  = os.path.join(BASE, "figures")
RES  = os.path.join(BASE, "results")
for d in [FIG, RES]:
    os.makedirs(d, exist_ok=True)
print("processed files:", sorted(os.listdir(PROC)))

## 0. Load the cleaned data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
PAL  = {"Healthy": "#2a9d8f", "Other_ICU": "#8d99ae", "Sepsis": "#e63946"}
MARS = {"Mars1": "#264653", "Mars2": "#e76f51", "Mars3": "#2a9d8f", "Mars4": "#e9c46a"}
OUTC = {"Died": "#e63946", "Survived": "#457b9d"}

pat   = pd.read_csv(os.path.join(PROC, "samples_clean.csv"), index_col=0, parse_dates=["run_date"])
genes = pd.read_csv(os.path.join(PROC, "genes_x_samples.csv"), index_col=0)            # all 802, all genes
E     = pd.read_csv(os.path.join(PROC, "genes_x_sepsis479_expressed.csv"), index_col=0) # 479 sepsis, expressed genes

sep = pat[pat.group == "Sepsis"].copy()
sep["died"] = sep.mortality_event_28days.astype(int)
sep["outcome"] = sep.died.map({0: "Survived", 1: "Died"})
sep["run_date_s"] = sep.run_date.dt.strftime("%Y-%m-%d")
print(pat.shape, genes.shape, E.shape)

Expected: `(802, 28) (11518, 802) (8200, 479)`.

## 1. Who is in the study? (Table 1)
We report **median (IQR)** for age because age is skewed (most patients are old, with a tail of younger ones).

In [ ]:
def describe(d):
    return pd.Series({"n": len(d), "age_median": d.age.median(),
                      "age_IQR": f"{d.age.quantile(.25):.0f}-{d.age.quantile(.75):.0f}",
                      "male_%": round(100 * d.male.mean(), 1)})
table1 = pat.groupby("group")[["age", "male"]].apply(describe)
table1.to_csv(os.path.join(RES, "table1_cohort.csv"))
print(f"Sepsis 28-day deaths: {sep.died.sum()} / {len(sep)} ({100*sep.died.mean():.1f}%)")
table1

Expected: Healthy 42 (median age 37), Other_ICU 281 (64), Sepsis 479 (63); 114 deaths (23.8%).
Healthy controls are ~25 years younger: keep this in mind for every sepsis-vs-healthy comparison.

## 2. What is associated with 28-day death? (sepsis patients only)
One test per variable, each with null hypothesis H0 = "this variable is unrelated to death":
- **Age** (number): Welch t-test (compares means) and Mann–Whitney U (compares ranks; doesn't need normal data).
- **Categories**: Fisher's exact test for 2×2 tables, chi-square for bigger tables.
- **Platelet score** (ordered 0–3): Mann–Whitney U.

In [ ]:
rows = []
def add(var, test, stat, p, detail):
    rows.append({"variable": var, "test": test, "statistic": round(float(stat), 3), "p": p, "detail": detail})

a, b = sep.age[sep.died == 1], sep.age[sep.died == 0]
r = stats.ttest_ind(a, b, equal_var=False)
add("age", "Welch t", r.statistic, r.pvalue, f"mean died {a.mean():.1f} vs survived {b.mean():.1f}")
r = stats.mannwhitneyu(a, b)
add("age", "Mann-Whitney U", r.statistic, r.pvalue, f"median died {a.median():.0f} vs survived {b.median():.0f}")

def cat_test(var, data=sep, label=None):
    ct = pd.crosstab(data[var], data.died)
    rate = (ct[1] / ct.sum(axis=1) * 100).round(1).to_dict()
    if ct.shape == (2, 2):
        o, p = stats.fisher_exact(ct)
        add(label or var, "Fisher exact", o, p, f"death % {rate}; n={int(ct.values.sum())}")
    else:
        c, p, _, exp = stats.chi2_contingency(ct)
        add(label or var, "chi-square", c, p, f"death % {rate}; n={int(ct.values.sum())}; min expected {exp.min():.1f}")

cat_test("gender", label="sex")
cat_test("diabetes_mellitus", label="diabetes")
cat_test("hospital")
cat_test("endotype_class", label="Mars endotype")
cat_test("infection_source", data=sep[sep.infection_source != "pneumonia+abdominal"])
cat_test("icu_acquired_infection", data=sep.dropna(subset=["icu_acquired_infection"]), label="ICU-acquired infection")
t = sep.dropna(subset=["thrombocytopenia_score"])
r = stats.mannwhitneyu(t.thrombocytopenia_score[t.died == 1], t.thrombocytopenia_score[t.died == 0])
add("platelet score (0 very low - 3 normal)", "Mann-Whitney U", r.statistic, r.pvalue,
    f"n={len(t)}; median died {t.thrombocytopenia_score[t.died==1].median()} vs survived {t.thrombocytopenia_score[t.died==0].median()}")
cat_test("run_date_s", label="array run date (batch)")

assoc = pd.DataFrame(rows)
assoc["p"] = assoc.p.map(lambda v: float(f"{v:.3g}"))
assoc.to_csv(os.path.join(RES, "mortality_associations.csv"), index=False)
pd.set_option("display.max_colwidth", 120)
assoc

Expected (p-values):

| Variable | p | Reading |
|---|---|---|
| age | 0.002 (Welch), 0.007 (MWU) | **Non-survivors older** (64.5 vs 59.9 y) |
| sex | 0.52 | no link |
| diabetes | 0.67 | no link (n = 390, complete cases) |
| hospital | 0.20 | no link |
| **Mars endotype** | **0.011** | **Mars1 34% vs 18–22%** |
| infection source | 0.43 | no link |
| ICU-acquired infection | 0.086 | trend only (33% vs 21%) |
| platelet score | 0.37 | no link, but only 95 patients have it |
| run date (batch) | 0.48 | batch is **not** related to death (some expected counts < 5, so treat as rough) |

### Age gradient and a combined model
Does risk climb steadily with age? And does Mars1 still matter once age, sex and hospital are in the same **logistic regression**? `exp(coefficient)` = **odds ratio (OR)**: OR > 1 means higher odds of death.

In [ ]:
sep["age_band"] = pd.cut(sep.age, [0, 50, 60, 70, 80, 100], labels=["<50", "50-59", "60-69", "70-79", "80+"])
print("death rate by age band:")
print(sep.groupby("age_band", observed=True).died.agg(n="size", death_rate="mean").round(3).to_string())

sep["Mars1"] = (sep.endotype_class == "Mars1").astype(int)
m = smf.logit("died ~ age + male + Mars1 + C(hospital)", data=sep).fit(disp=0)
orr = pd.DataFrame({"OR": np.exp(m.params), "CI_low": np.exp(m.conf_int()[0]),
                    "CI_high": np.exp(m.conf_int()[1]), "p": m.pvalues}).drop("Intercept").round(3)
orr.to_csv(os.path.join(RES, "logistic_age_sex_mars1_hospital.csv"))
orr

Expected: death rate 14% (<50) → 28% → 21% → 31% → 33% (80+). Logistic: **Mars1 OR 2.05 (1.31–3.23), p = 0.002**; age OR 1.023 per year (p = 0.005); sex and hospital not significant.

**Reading:** each extra year adds ~2.3% to the odds of death; being Mars1 doubles the odds, and that is **not explained** by age, sex or hospital.

## 3. Genes vs death
For each of the 8,200 expressed genes we run a **Welch t-test** (died vs survived). With 8,200 tests, ~410 would be "significant" at p < 0.05 by pure chance, so we control the **false discovery rate (FDR)** with Benjamini–Hochberg and call genes with FDR < 0.05 significant.
`log2FC` = mean(died) − mean(survived); because the data are log2, log2FC = 1 means twice as high.

In [ ]:
def de(E, mask):
    """Welch t-test for every gene: group 'mask' vs the rest, with BH-FDR."""
    t_, p_ = stats.ttest_ind(E.loc[:, mask], E.loc[:, ~mask], axis=1, equal_var=False)
    r = pd.DataFrame({"log2FC": E.loc[:, mask].mean(axis=1) - E.loc[:, ~mask].mean(axis=1), "t": t_, "p": p_},
                     index=E.index)
    r["FDR"] = multipletests(r.p, method="fdr_bh")[1]
    return r.sort_values("p")

died_mask = (sep.loc[E.columns, "died"] == 1).values
de_death = de(E, died_mask)
de_death.to_csv(os.path.join(RES, "DE_died_vs_survived.csv"))
sig = de_death[de_death.FDR < 0.05]
print(f"{len(sig)} genes FDR<0.05 of {len(E)}: {(sig.log2FC < 0).sum()} lower, {(sig.log2FC > 0).sum()} higher in non-survivors; "
      f"largest |log2FC| {sig.log2FC.abs().max():.2f}")
print("top lower in died: ", sig[sig.log2FC < 0].head(8).index.tolist())
print("top higher in died:", sig[sig.log2FC > 0].head(8).index.tolist())

Expected: **680 genes**, 564 lower and 116 higher in non-survivors; largest |log2FC| 0.97 (< 2-fold).
Top lower: TNFSF12, TRIM27, TNFRSF8, **CX3CR1**, AMICA1 …, mostly lymphocyte / NK-cell genes, i.e. the **immune-suppression** picture. Top higher: TUBG2, NCRNA00188, TMCC2 …

### Sensitivity check: drop the flagged samples
Re-run without the samples flagged in notebook 01 (sex mismatch, unusual arrays, early censoring). If the results are real, the gene ranking should barely change.

In [ ]:
flag = (sep.flag_sex_mismatch | sep.flag_expr_outlier | sep.flag_censored_early).reindex(E.columns).values
de_s = de(E.loc[:, ~flag], died_mask[~flag])
fl = sep.loc[E.columns[flag]]
print(f"without {flag.sum()} flagged samples ({fl.died.sum()} died, {(fl.endotype_class == 'Mars1').sum()} Mars1): "
      f"{(de_s.FDR < 0.05).sum()} genes FDR<0.05, all inside the main list: "
      f"{set(de_s.index[de_s.FDR < 0.05]) <= set(sig.index)}")
print(f"correlation of t-statistics, full vs sensitivity: r = "
      f"{np.corrcoef(de_death.t, de_s.t.reindex(de_death.index))[0, 1]:.3f}")

Expected: 12 samples removed (5 died, 7 Mars1) → 251 genes, all inside the main 680; t-statistics r = 0.984.

**Reading:** the list shrinks because the removed samples are the strongest-signal patients (mostly Mars1 non-survivors), not because results flip. The ranking is essentially unchanged, so the finding is **robust**.

### Sepsis vs healthy, and genes that track age

In [ ]:
H = genes.loc[E.index, pat.index[pat.group == "Healthy"]]
de_sh = de(pd.concat([E, H], axis=1), np.r_[np.ones(E.shape[1], bool), np.zeros(H.shape[1], bool)])
de_sh.to_csv(os.path.join(RES, "DE_sepsis_vs_healthy.csv"))
s2 = de_sh[(de_sh.FDR < 0.05) & (de_sh.log2FC.abs() > 1)]
print(f"sepsis vs healthy: {(de_sh.FDR < 0.05).sum()} genes FDR<0.05, {len(s2)} also >2-fold")
print("top up in sepsis:  ", s2.sort_values("log2FC", ascending=False).head(8).index.tolist())
print("top down in sepsis:", s2.sort_values("log2FC").head(8).index.tolist())

rho = E.T.corrwith(sep.loc[E.columns, "age"], method="spearman").sort_values()
print("\nstrongest age correlations (Spearman rho):", rho.head(3).round(2).to_dict(), rho.tail(3).round(2).to_dict())

Expected: 7,007 genes FDR < 0.05, **1,131 more than 2-fold**. Up in sepsis: CD177, ARG1, HP, RETN, MMP8 … (neutrophil / emergency-granulopoiesis genes). Down: FCER1A, BCL11B, IL2RB, IL7R, ITK … (T cells and other lymphocytes).
Age correlations are weak (|rho| ≤ 0.24): age and gene expression carry **different** information, which is why the manual later asks whether genes beat age at predicting death.

**Contrast:** sepsis vs healthy = huge changes (> 2-fold); died vs survived = many small shifts (< 2-fold).

## 4. Structure: PCA, and is batch a problem?
PCA squeezes the 2,000 most variable genes into a few axes (principal components) that capture the most variation. Each gene is z-scored first so highly expressed genes don't dominate.
To test batch, we ask how much of each axis is explained by array run date (R² from a one-way model). Small R² = batch is not driving the structure.

In [ ]:
def pca(df, n=10):
    top = df.var(axis=1).sort_values(ascending=False).index[:2000]
    p = PCA(n_components=n, random_state=0)
    return p.fit_transform(StandardScaler().fit_transform(df.loc[top].T)), p.explained_variance_ratio_

P_all, v_all = pca(genes)
P_sep, v_sep = pca(genes[sep.index])
sep["PC1"], sep["PC2"] = P_sep[:, 0], P_sep[:, 1]
print(f"variance on PC1+PC2: all samples {100*v_all[:2].sum():.0f}%, sepsis only {100*v_sep[:2].sum():.0f}%")
for pc in ["PC1", "PC2"]:
    print(f"sepsis {pc}: Spearman rho with age {stats.spearmanr(sep[pc], sep.age)[0]:.2f}; "
          f"died vs survived Mann-Whitney p = {stats.mannwhitneyu(sep[pc][sep.died == 1], sep[pc][sep.died == 0])[1]:.3g}")
r2 = [smf.ols("y ~ C(d)", data=pd.DataFrame({"y": P_sep[:, i], "d": sep.run_date_s.values})).fit().rsquared
      for i in range(5)]
print("R^2 of run date on sepsis PC1-5:", np.round(r2, 3).tolist())

Expected: PC1+PC2 = 37% (all) and 35% (sepsis); PC2 differs between died and survived (p = 0.0005) while PC1 does not; run date explains **at most 2.5%** of any PC.
**Reading:** there is strong structure, it relates to death along PC2, it is not age, and it is **not batch**.

## 5. Figures
Each figure is saved to `figures/` at 200 dpi (change `dpi=300` for the final report).

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(19, 4.3))
sns.boxplot(data=pat, x="group", y="age", order=list(PAL), palette=PAL, hue="group", legend=False, ax=ax[0])
ax[0].set(title="A  Age by group", xlabel="")

sns.histplot(data=sep, x="age", hue="outcome", bins=25, kde=True, ax=ax[1], palette=OUTC)
ax[1].set(title=f"B  Age by 28-day outcome (MWU p={stats.mannwhitneyu(a, b)[1]:.3f})")

ar = sep.groupby("age_band", observed=True).died.mean() * 100
ax[2].bar(ar.index.astype(str), ar.values, color="#e63946")
for i, (v, n) in enumerate(zip(ar.values, sep.age_band.value_counts().reindex(ar.index).values)):
    ax[2].text(i, v + 0.8, f"{v:.0f}%\nn={n}", ha="center", fontsize=8)
ax[2].set(title="C  Death rate by age band", ylabel="28-day mortality (%)", xlabel="age", ylim=(0, 45))

mr = sep.groupby("endotype_class").died.mean() * 100
ax[3].bar(mr.index, mr.values, color=[MARS[k] for k in mr.index])
for i, (k, v) in enumerate(mr.items()):
    ax[3].text(i, v + 0.8, f"{v:.0f}%\nn={(sep.endotype_class == k).sum()}", ha="center", fontsize=8)
p_mars = assoc.loc[assoc.variable == "Mars endotype", "p"].iloc[0]
ax[3].set(title=f"D  Death rate by Mars endotype (chi2 p={p_mars:.3f})", ylabel="28-day mortality (%)", ylim=(0, 45))
plt.tight_layout(); plt.savefig(os.path.join(FIG, "02_cohort_mortality.png"), dpi=200); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8))
d = pd.DataFrame(P_all[:, :2], index=genes.columns, columns=["PC1", "PC2"]).join(pat.group)
sns.scatterplot(data=d, x="PC1", y="PC2", hue="group", palette=PAL, s=10, ax=ax[0], linewidth=0)
ax[0].set(title=f"A  All 802 samples ({100*v_all[0]:.0f}% / {100*v_all[1]:.0f}% variance)")
sns.scatterplot(data=sep, x="PC1", y="PC2", hue="endotype_class", palette=MARS, hue_order=list(MARS),
                s=12, ax=ax[1], linewidth=0)
ax[1].set(title="B  Sepsis only, by Mars endotype")
sns.scatterplot(data=sep.sort_values("died"), x="PC1", y="PC2", hue="outcome",
                palette={"Died": "#e63946", "Survived": "#c9d6df"}, s=12, ax=ax[2], linewidth=0)
ax[2].set(title="C  Sepsis only, by 28-day outcome")
plt.tight_layout(); plt.savefig(os.path.join(FIG, "02_pca.png"), dpi=200); plt.show()

**What you should see:** (A) healthy controls in their own group at the top; sepsis and other-ICU patients overlap in one cloud. (B) No separate islands, but an **ordered** cloud: Mars2 left, Mars3 top, Mars1 bottom-right. (C) Deaths everywhere, somewhat more at the bottom (Mars1 region).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"width_ratios": [1.3, 1, 1.3]})
c = np.where(de_death.FDR >= 0.05, "#d9d9d9", np.where(de_death.log2FC > 0, "#e63946", "#457b9d"))
ax[0].scatter(de_death.log2FC, -np.log10(de_death.p), c=c, s=5)
for gname in ["CX3CR1", "TNFSF12", "TNFRSF8", "AMICA1", "TUBG2", "NCRNA00188"]:
    ax[0].annotate(gname, (de_death.log2FC[gname], -np.log10(de_death.p[gname])), fontsize=7,
                   xytext=(4, -3), textcoords="offset points")
ax[0].set(title=f"A  Died vs survived: {len(sig)} genes FDR<0.05",
          xlabel="log2 fold change (died - survived)", ylabel="-log10 p")

ax[1].hist(de_death.p, bins=40, color="#8d99ae")
ax[1].set(title="B  p-value histogram (spike = real signal)", xlabel="p")

show = sig[sig.log2FC < 0].index[:3].tolist() + sig[sig.log2FC > 0].index[:3].tolist()
long = E.loc[show].T.join(sep.outcome).melt(id_vars="outcome", var_name="gene", value_name="log2 expression")
sns.boxplot(data=long, x="gene", y="log2 expression", hue="outcome", palette=OUTC, ax=ax[2], fliersize=1)
ax[2].set(title="C  Top genes lower / higher in non-survivors", xlabel="")
ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "02_genes_death.png"), dpi=200); plt.show()

**Volcano (A):** x = effect size, y = significance; blue = lower in non-survivors (many more), red = higher. **Histogram (B):** a flat floor (genes with no real difference) plus a tall spike near 0 (real signal).

### Kaplan–Meier survival by Mars endotype
The fraction still alive over 28 days. It handles **censoring**: the 2 patients followed only 5–8 days count as "at risk" until they leave, instead of being counted as survivors. Written by hand here (product-limit formula); the manual's Step 6 does the same with the `lifelines` library and adds the log-rank test.

In [ ]:
def km(time, event):
    """Kaplan-Meier: at each death time, multiply survival by (1 - deaths / number at risk)."""
    s, out = 1.0, [(0, 1.0)]
    for t_ in np.sort(time[event == 1].unique()):
        at_risk = (time >= t_).sum()
        d_ = ((time == t_) & (event == 1)).sum()
        s *= 1 - d_ / at_risk
        out.append((t_, s))
    out.append((28, s))
    return np.array(out, dtype=float)

plt.figure(figsize=(6.5, 4.6))
for k, col in MARS.items():
    dd = sep[sep.endotype_class == k]
    c_ = km(dd.time_to_event_28days, dd.died)
    plt.step(c_[:, 0], c_[:, 1], where="post", color=col, lw=2,
             label=f"{k} (n={len(dd)}, {100*dd.died.mean():.0f}% died)")
plt.ylim(0.55, 1.01); plt.xlabel("days since ICU admission"); plt.ylabel("fraction alive")
plt.title("Kaplan-Meier survival by Mars endotype"); plt.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(FIG, "02_km_mars.png"), dpi=200); plt.show()

**What you should see:** Mars1 drops fastest, mostly in the first 1–2 weeks, ending at ~66% alive; the other three end at ~78–82%.

## Summary of patterns
1. **Age** is the only clinical variable linked to death (older = higher risk). Sex, diabetes, hospital, infection source and platelet score are not.
2. **Endotype** matters: Mars1 has 34% mortality and doubles the odds of death even after adjusting for age, sex and hospital (OR 2.05).
3. **Genes:** 680 genes differ between non-survivors and survivors (FDR < 0.05), mostly **lower** lymphocyte / NK genes, and all differences are small (< 2-fold). The ranking is robust to removing flagged samples.
4. **Sepsis vs healthy** is a massive change: neutrophil genes up, T-cell genes down.
5. **Structure:** sepsis patients form one continuous cloud ordered by endotype, not separate islands. PC2 relates to death; age and batch do not drive the structure.

**Next:** the manual's Step 5 (clustering) and Steps 6–11 can now load the cleaned files from `data/processed/` directly.